Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# @title 1) ComfyUI Kurulum

from pathlib import Path

OPTIONS = {
    'USE_GOOGLE_DRIVE': False,
    'UPDATE_COMFY_UI': True,
    'USE_COMFYUI_MANAGER': True,
    'INSTALL_CUSTOM_NODES_DEPENDENCIES': True,
}

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

# ComfyUI klonla (yoksa)
![ ! -d $WORKSPACE ] && echo "-= Initial setup ComfyUI =-" && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

# Güncelle
if OPTIONS['UPDATE_COMFY_UI']:
    !echo "-= Updating ComfyUI =-"
    !git pull

# ComfyUI'nin kendi requirements.txt'ini kur (en önemli kısım — comfy_aimdo vs. yeni modüller için)
!echo "-= Installing ComfyUI requirements.txt =-"
!pip install -r requirements.txt

# Ek paketler (güvenlik için, bazıları requirements.txt'te olmayabilir)
!pip install accelerate torchsde kornia>=0.7.1 spandrel soundfile sentencepiece

# ComfyUI-Manager kur
if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd custom_nodes
    ![ ! -d ComfyUI-Manager ] && echo "-= Setup ComfyUI-Manager =-" && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull
    %cd $WORKSPACE

# Manager varsa onun dependency restore'unu çalıştır (idempotent, zararı olmaz)
if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
    !echo "-= Install custom nodes dependencies =-"
    !pip install GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies 2>/dev/null || echo "(Manager cm-cli atlandı — önemli değil)"

print("\n✅ ComfyUI kurulumu tamamlandı")

-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 36721, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 36721 (delta 107), reused 83 (delta 83), pack-reused 36591 (from 4)
Receiving objects: 100% (36721/36721), 79.39 MiB | 19.17 MiB/s, done.
Resolving deltas: 100% (24602/24602), done.
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Installing ComfyUI requirements.txt =-
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [ ]:
# @title 2) Proje Hazırlık: Custom Nodes + Wan 2.1 VACE Modelleri

import os

# ============================================================
# BÖLÜM A — Custom Nodes (3 adet)
# ============================================================
print("=" * 60)
print("CUSTOM NODES KURULUMU")
print("=" * 60)

os.chdir('/content/ComfyUI/custom_nodes')

# 1. VideoHelperSuite — video yükleme/kaydetme
![ ! -d ComfyUI-VideoHelperSuite ] && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# 2. KJNodes — Kijai utility nodes (Wan workflow'ları için gerekli)
![ ! -d ComfyUI-KJNodes ] && git clone https://github.com/kijai/ComfyUI-KJNodes.git

# 3. ComfyUI_essentials — cubiq utilities
![ ! -d ComfyUI_essentials ] && git clone https://github.com/cubiq/ComfyUI_essentials.git

# 4. ComfyUI_LayerStyle — chflame163 (LayerUtility: ColorImage V2 vb.)
![ ! -d ComfyUI_LayerStyle ] && git clone https://github.com/chflame163/ComfyUI_LayerStyle.git

# Her node'un kendi bağımlılıklarını kur
print("\n--- Installing node dependencies ---")
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt
!pip install -r ComfyUI-KJNodes/requirements.txt
!pip install -r ComfyUI_essentials/requirements.txt
!pip install -r ComfyUI_LayerStyle/requirements.txt

os.chdir('/content/ComfyUI')
print("✅ Custom nodes hazır\n")

# ============================================================
# BÖLÜM B — Wan 2.1 VACE Modelleri (~42 GB, aria2c ile paralel)
# ============================================================
print("=" * 60)
print("MODEL İNDİRME (~42 GB)")
print("=" * 60)

!apt-get install -y -qq aria2

os.makedirs('/content/ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('/content/ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('/content/ComfyUI/models/vae', exist_ok=True)
os.makedirs('/content/ComfyUI/models/loras', exist_ok=True)

ARIA = "aria2c -x 16 -s 16 -c -k 1M --console-log-level=warn --summary-interval=10"

print("\n[1/4] Wan 2.1 VACE 14B fp16 (~34.7 GB)")
!{ARIA} -d /content/ComfyUI/models/diffusion_models \
    -o wan2.1_vace_14B_fp16.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_vace_14B_fp16.safetensors"

print("\n[2/4] UMT5-XXL fp8 text encoder (~6.7 GB)")
!{ARIA} -d /content/ComfyUI/models/text_encoders \
    -o umt5_xxl_fp8_e4m3fn_scaled.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"

print("\n[3/4] Wan 2.1 VAE (~0.5 GB)")
!{ARIA} -d /content/ComfyUI/models/vae \
    -o wan_2.1_vae.safetensors \
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors"

print("\n[4/4] LightX2V LoRA rank32 (~0.6 GB)")
!{ARIA} -d /content/ComfyUI/models/loras \
    -o Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors \
    "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors"

# Doğrulama
print("\n" + "=" * 60)
print("KONTROL — Dosya Boyutları")
print("=" * 60)
!ls -lh /content/ComfyUI/models/diffusion_models/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/
!ls -lh /content/ComfyUI/models/loras/

print("\n✅ Proje hazırlığı tamamlandı")

CUSTOM NODES KURULUMU
Cloning into 'ComfyUI-VideoHelperSuite'...
remote: Enumerating objects: 3355, done.
remote: Counting objects: 100% (1598/1598), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 3355 (delta 1463), reused 1232 (delta 1229), pack-reused 1757 (from 3)
Receiving objects: 100% (3355/3355), 792.82 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (1978/1978), done.
Cloning into 'ComfyUI-KJNodes'...
remote: Enumerating objects: 4850, done.
remote: Counting objects: 100% (1733/1733), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 4850 (delta 1560), reused 1463 (delta 1463), pack-reused 3117 (from 2)
Receiving objects: 100% (4850/4850), 25.70 MiB | 16.44 MiB/s, done.
Resolving deltas: 100% (3318/3318), done.
Cloning into 'ComfyUI_essentials'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 480 (delta 125), reused 114 (delta 114)

### Run ComfyUI with cloudflared (Recommended Way)




In [ ]:
# @title 3) ComfyUI'yi Başlat (Cloudflared Tunnel, Arka Planda, Akıllı Bekleme)

!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

# Cloudflared tüneli başlatacak thread
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n✅ ComfyUI yüklendi, Cloudflared tüneli başlatılıyor...\n")

    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ComfyUI'yi arka planda başlat (kernel'i bloklama)
subprocess.Popen(["python", "main.py", "--dont-print-server"])

# ============================================================
# AKILLI BEKLEME — ComfyUI API gerçekten hazır olana kadar bekle
# ============================================================
print("⏳ ComfyUI'nin API'sinin ayağa kalkması bekleniyor...")

API_URL       = "http://127.0.0.1:8188/system_stats"
MAX_WAIT_SEC  = 300   # 5 dakika (büyük modeller yüklenirken uzun sürebilir)
CHECK_EVERY   = 3     # saniye

start = time.time()
ready = False
while time.time() - start < MAX_WAIT_SEC:
    try:
        with urllib.request.urlopen(API_URL, timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    elapsed = int(time.time() - start)
    print(f"   ... {elapsed}s (henüz hazır değil)")
    time.sleep(CHECK_EVERY)

if ready:
    total = int(time.time() - start)
    print(f"\n✅ ComfyUI API hazır ({total} saniyede)")
    print("➡️  Hücre 4'ü çalıştırabilirsin")
else:
    print(f"\n❌ ComfyUI {MAX_WAIT_SEC} saniyede hazır olmadı")
    print("   Muhtemel sebepler: model yüklenme hatası, disk dolu, vs.")
    print("   Colab log'larını kontrol et veya runtime'ı yeniden başlat")

--2026-04-24 17:19:09--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-24 17:19:10--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-24T17%3A56%3A24Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b

In [ ]:
# @title 4) Batch Processor — Drive'daki videoları sırayla işle (recursive)
#
# ÖN HAZIRLIK (Drive'da):
#   /MyDrive/wan_batch/workflow.json              ← API-format workflow
#   /MyDrive/wan_batch/input_videos/              ← videolar (iç içe klasör olabilir)
#   /MyDrive/wan_batch/output_videos/             ← script dolduracak (yapı korunur)
#   /MyDrive/wan_batch/batch_log.txt              ← script yazacak
#
# Özellikler:
#   - Recursive: iç içe klasörlerdeki videoları bulur
#   - Klasör yapısı korunur: input/A/B/vid.mp4 → output/A/B/vid_out.mp4
#   - Süre filtresi: MAX_VIDEO_DURATION_SEC'den uzun olanları atlar
#   - Resumable: Drive'da zaten olanı skip eder (crash recovery)
#   - Her video bitince Drive'a yazar
#   - Hata olursa atlar, log'a yazar, devam eder
#   - Video başına timeout: 30 dakika

import os
import json
import time
import shutil
import uuid
import traceback
import subprocess
import urllib.request
import urllib.error
from datetime import datetime
from pathlib import Path

# ============================================================
# AYARLAR
# ============================================================
DRIVE_ROOT           = "/content/drive/MyDrive/wan_batch"
WORKFLOW_PATH        = f"{DRIVE_ROOT}/workflow.json"
INPUT_DIR_DRIVE      = f"{DRIVE_ROOT}/input_videos"
OUTPUT_DIR_DRIVE     = f"{DRIVE_ROOT}/output_videos"
LOG_PATH             = f"{DRIVE_ROOT}/batch_log.txt"

COMFY_INPUT_DIR      = "/content/ComfyUI/input"
COMFY_OUTPUT_DIR     = "/content/ComfyUI/output"
COMFY_API            = "http://127.0.0.1:8188"

POLL_INTERVAL        = 10         # saniye
TIMEOUT_PER_VIDEO    = 30 * 60    # 30 dakika

# Video süresi limiti — bundan uzun videolar atlanır
MAX_VIDEO_DURATION_SEC = 15.0     # saniye (None yaparsan kontrol kapanır)

VIDEO_EXTS = (".mp4", ".mov", ".webm", ".mkv", ".avi")

# ============================================================
# DRIVE MOUNT (sadece I/O için, ComfyUI Drive görmez)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)

# ============================================================
# LOG YARDIMCISI
# ============================================================
def log(msg):
    """Hem ekrana hem Drive'daki log dosyasına yaz."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

# ============================================================
# COMFYUI API YARDIMCILARI
# ============================================================
CLIENT_ID = str(uuid.uuid4())

def api_get(endpoint):
    req = urllib.request.Request(f"{COMFY_API}{endpoint}")
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())

def api_post(endpoint, data):
    body = json.dumps(data).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFY_API}{endpoint}",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())

def check_comfy_ready():
    try:
        api_get("/system_stats")
        return True
    except Exception as e:
        log(f"❌ ComfyUI API yanıt vermiyor: {e}")
        return False

def queue_prompt(workflow_dict):
    payload = {"prompt": workflow_dict, "client_id": CLIENT_ID}
    resp = api_post("/prompt", payload)
    return resp["prompt_id"]

def wait_for_completion(prompt_id, timeout):
    start = time.time()
    while True:
        if time.time() - start > timeout:
            raise TimeoutError(f"Prompt {prompt_id} timeout ({timeout}s)")
        try:
            history = api_get(f"/history/{prompt_id}")
            if prompt_id in history:
                return history[prompt_id]
        except Exception:
            pass
        time.sleep(POLL_INTERVAL)

# ============================================================
# VIDEO YARDIMCILARI
# ============================================================
def get_video_duration(path):
    """ffprobe ile video süresini (saniye) döndür, başarısızsa None."""
    try:
        result = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ],
            capture_output=True, text=True, timeout=30,
        )
        if result.returncode == 0:
            return float(result.stdout.strip())
    except Exception:
        pass
    return None

def scan_videos_recursive(root_dir):
    """
    Recursive tarama. Her video için:
      (absolute_path, relative_path) tuple döndür.
    relative_path input root'a göre ('klip_2024/video.mp4' gibi).
    """
    found = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith(VIDEO_EXTS):
                abs_path = os.path.join(dirpath, fn)
                rel_path = os.path.relpath(abs_path, root_dir)
                found.append((abs_path, rel_path))
    found.sort(key=lambda x: x[1])
    return found

def unique_comfy_filename(rel_path):
    """
    Relative path'i benzersiz flat isme çevir.
    'klip_2024/video.mp4' → 'klip_2024__video.mp4'
    'a/b/c/vid.mp4'       → 'a__b__c__vid.mp4'
    """
    parts = rel_path.replace("\\", "/").split("/")
    return "__".join(parts)

# ============================================================
# WORKFLOW YARDIMCILARI
# ============================================================
def load_workflow_template():
    with open(WORKFLOW_PATH, "r", encoding="utf-8") as f:
        wf = json.load(f)
    if "nodes" in wf:
        raise ValueError(
            "workflow.json UI formatında görünüyor. ComfyUI'de "
            "'Workflow → Export (API)' veya 'Save (API Format)' ile kaydet."
        )
    return wf

def set_input_video(workflow_dict, video_filename):
    """API formatta VHS_LoadVideo node'unun video alanını güncelle."""
    changed = False
    for node_id, node in workflow_dict.items():
        if node.get("class_type") == "VHS_LoadVideo":
            if "inputs" in node:
                node["inputs"]["video"] = video_filename
                changed = True
    if not changed:
        raise RuntimeError("Workflow'da VHS_LoadVideo node bulunamadı")
    return workflow_dict

def find_output_video(history, fallback_dir=COMFY_OUTPUT_DIR):
    """History'den üretilen video dosyasının yolunu bul."""
    outputs = history.get("outputs", {})
    for node_id, out in outputs.items():
        for key in ("gifs", "images", "videos"):
            if key in out:
                for item in out[key]:
                    filename = item.get("filename")
                    subfolder = item.get("subfolder", "")
                    if filename and filename.lower().endswith((".mp4", ".webm", ".mov")):
                        full_path = os.path.join(fallback_dir, subfolder, filename)
                        if os.path.exists(full_path):
                            return full_path
    # Fallback: /output/ içindeki en yeni .mp4
    mp4s = []
    for root, _, files in os.walk(fallback_dir):
        for fn in files:
            if fn.lower().endswith(".mp4"):
                fp = os.path.join(root, fn)
                mp4s.append((os.path.getmtime(fp), fp))
    if mp4s:
        mp4s.sort(reverse=True)
        return mp4s[0][1]
    return None

# ============================================================
# ANA DÖNGÜ
# ============================================================
def main():
    log("=" * 60)
    log("BATCH PROCESSING BAŞLIYOR")
    log("=" * 60)

    if not check_comfy_ready():
        log("❌ ComfyUI API hazır değil. Önce Hücre 3'ü çalıştır.")
        return

    try:
        template = load_workflow_template()
        log(f"✅ Workflow yüklendi: {WORKFLOW_PATH}")
    except Exception as e:
        log(f"❌ Workflow yüklenemedi: {e}")
        return

    if not os.path.isdir(INPUT_DIR_DRIVE):
        log(f"❌ Input klasörü yok: {INPUT_DIR_DRIVE}")
        return

    videos = scan_videos_recursive(INPUT_DIR_DRIVE)
    if not videos:
        log(f"❌ Input klasöründe video yok: {INPUT_DIR_DRIVE}")
        return

    log(f"📁 Bulunan video sayısı: {len(videos)} (recursive tarama)")
    if MAX_VIDEO_DURATION_SEC:
        log(f"⏱️  Maksimum video süresi: {MAX_VIDEO_DURATION_SEC}s (bundan uzunlar atlanır)")

    stats = {"total": len(videos), "ok": 0, "skipped_exists": 0,
             "skipped_duration": 0, "failed": 0}

    for idx, (abs_path, rel_path) in enumerate(videos, 1):
        # Orijinal yapıyı koruyarak output yolunu hesapla
        rel_dir  = os.path.dirname(rel_path)
        stem     = Path(rel_path).stem
        output_rel_path  = os.path.join(rel_dir, f"{stem}_out.mp4") if rel_dir else f"{stem}_out.mp4"
        output_abs_drive = os.path.join(OUTPUT_DIR_DRIVE, output_rel_path)

        log(f"\n--- [{idx}/{len(videos)}] {rel_path} ---")

        # Resume: zaten var mı?
        if os.path.exists(output_abs_drive) and os.path.getsize(output_abs_drive) > 0:
            log(f"⏭️  SKIP (zaten var): {output_rel_path}")
            stats["skipped_exists"] += 1
            continue

        # Süre kontrolü
        if MAX_VIDEO_DURATION_SEC:
            duration = get_video_duration(abs_path)
            if duration is None:
                log(f"⚠️  Süre okunamadı, yine de deniyorum")
            elif duration > MAX_VIDEO_DURATION_SEC:
                log(f"⏭️  SKIP (çok uzun: {duration:.1f}s > {MAX_VIDEO_DURATION_SEC}s)")
                stats["skipped_duration"] += 1
                continue

        t_start = time.time()
        comfy_filename = unique_comfy_filename(rel_path)
        comfy_input_path  = os.path.join(COMFY_INPUT_DIR, comfy_filename)
        output_path_comfy = None

        try:
            # 1) Drive → Colab (benzersiz isimle)
            os.makedirs(COMFY_INPUT_DIR, exist_ok=True)
            shutil.copy2(abs_path, comfy_input_path)
            log(f"  📥 Kopyalandı → {comfy_filename}")

            # 2) Workflow hazırla
            wf = json.loads(json.dumps(template))  # deep copy
            wf = set_input_video(wf, comfy_filename)

            # 3) Queue
            prompt_id = queue_prompt(wf)
            log(f"  🚀 Queue edildi: prompt_id={prompt_id}")

            # 4) Bekle
            history = wait_for_completion(prompt_id, TIMEOUT_PER_VIDEO)

            # Execution hatası var mı?
            status = history.get("status", {})
            if status.get("status_str") == "error":
                messages = status.get("messages", [])
                raise RuntimeError(f"ComfyUI execution error: {messages}")

            # 5) Output videoyu bul
            output_path_comfy = find_output_video(history)
            if not output_path_comfy or not os.path.exists(output_path_comfy):
                raise FileNotFoundError("Üretilen output videosu bulunamadı")

            # 6) Drive'a yaz (orijinal klasör yapısını koru)
            os.makedirs(os.path.dirname(output_abs_drive) or ".", exist_ok=True)
            shutil.copy2(output_path_comfy, output_abs_drive)

            size_mb = os.path.getsize(output_abs_drive) / 1024 / 1024
            elapsed = time.time() - t_start
            log(f"  ✅ OK → {output_rel_path} ({size_mb:.1f} MB, {elapsed:.0f}s)")
            stats["ok"] += 1

        except Exception as e:
            elapsed = time.time() - t_start
            log(f"  ❌ FAILED ({elapsed:.0f}s): {type(e).__name__}: {e}")
            last_tb = traceback.format_exc().splitlines()[-1]
            log(f"     {last_tb}")
            stats["failed"] += 1

        finally:
            # Colab diskini temizle (her durumda, disk dolmasın)
            try:
                if os.path.exists(comfy_input_path):
                    os.remove(comfy_input_path)
            except Exception:
                pass
            try:
                if output_path_comfy and os.path.exists(output_path_comfy):
                    os.remove(output_path_comfy)
            except Exception:
                pass

    # Özet
    log("\n" + "=" * 60)
    log("BATCH TAMAMLANDI")
    log(f"  Toplam:               {stats['total']}")
    log(f"  ✅ Başarılı:           {stats['ok']}")
    log(f"  ⏭️  Atlandı (var):      {stats['skipped_exists']}")
    log(f"  ⏭️  Atlandı (uzun):     {stats['skipped_duration']}")
    log(f"  ❌ Başarısız:          {stats['failed']}")
    log("=" * 60)


main()


✅ ComfyUI yüklendi, Cloudflared tüneli başlatılıyor...

🌐 ComfyUI URL: https://husband-thrown-before-pointed.trycloudflare.com                                   |
Mounted at /content/drive
[2026-04-24 17:20:04] ============================================================
[2026-04-24 17:20:06] BATCH PROCESSING BAŞLIYOR
[2026-04-24 17:20:06] ============================================================
[2026-04-24 17:20:07] ✅ Workflow yüklendi: /content/drive/MyDrive/wan_batch/workflow.json
[2026-04-24 17:20:08] 📁 Bulunan video sayısı: 15 (recursive tarama)
[2026-04-24 17:20:08] ⏱️  Maksimum video süresi: 15.0s (bundan uzunlar atlanır)
[2026-04-24 17:20:08] 
--- [1/15] 2/108665792.mp4 ---
[2026-04-24 17:20:11]   📥 Kopyalandı → 2__108665792.mp4
[2026-04-24 17:20:11]   🚀 Queue edildi: prompt_id=a269379c-ccc7-4d71-9288-b497b543f957
[2026-04-24 17:23:21]   ✅ OK → 2/108665792_out.mp4 (1.4 MB, 190s)
[2026-04-24 17:23:21] 
--- [2/15] 2/108670716.mp4 ---
[2026-04-24 17:23:22]   📥 Kopyalandı → 2_